In [1]:
import requests
import json
import os
from datetime import datetime
from bs4 import BeautifulSoup
import pandas as pd
print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
URL = "https://anilist.co/"
headers = {
    "User-Agent": "Anime Scrapper"
}

In [3]:
response = requests.get(URL, headers=headers)

In [4]:
print(response)

<Response [200]>


In [5]:
print(response.content)

b'<!DOCTYPE html><html lang=en><head><title>AniList</title><meta charset=utf-8><meta http-equiv=x-ua-compatible content="ie=edge"><meta http-equiv=Content-Security-Policy content=block-all-mixed-content><meta name=viewport content="width=device-width,initial-scale=1,maximum-scale=1,minimum-scale=1,user-scalable=no"><meta property=og:site_name content=AniList><meta name=twitter:site content=@AniListco><script>window.al_token = "9iAXwcVKJgY2aTYaVfoRoaDLtyYgL6Qvez6M6z4l";</script><link href="//fonts.googleapis.com/css?family=Roboto:300,400,500,700" rel=stylesheet><link rel=preload as=style href="https://fonts.googleapis.com/css?family=Overpass:400,600,700,800"><link href="https://fonts.googleapis.com/css?family=Overpass:400,600,700,800" rel=stylesheet><link rel=icon type=image/png sizes=32x32 href=/img/icons/favicon-32x32.png><link rel=icon type=image/png sizes=16x16 href=/img/icons/favicon-16x16.png><link rel=manifest href=/manifest.json><meta name=theme-color content=#2b2d42><meta name=

In [7]:
#Create a Beautiful Soup object
soup = BeautifulSoup(response.text, 'lxml')

In [8]:
print(soup)

<!DOCTYPE html>
<html lang="en"><head><title>AniList</title><meta charset="utf-8"/><meta content="ie=edge" http-equiv="x-ua-compatible"/><meta content="block-all-mixed-content" http-equiv="Content-Security-Policy"/><meta content="width=device-width,initial-scale=1,maximum-scale=1,minimum-scale=1,user-scalable=no" name="viewport"/><meta content="AniList" property="og:site_name"/><meta content="@AniListco" name="twitter:site"/><script>window.al_token = "9iAXwcVKJgY2aTYaVfoRoaDLtyYgL6Qvez6M6z4l";</script><link href="//fonts.googleapis.com/css?family=Roboto:300,400,500,700" rel="stylesheet"/><link as="style" href="https://fonts.googleapis.com/css?family=Overpass:400,600,700,800" rel="preload"/><link href="https://fonts.googleapis.com/css?family=Overpass:400,600,700,800" rel="stylesheet"/><link href="/img/icons/favicon-32x32.png" rel="icon" sizes="32x32" type="image/png"/><link href="/img/icons/favicon-16x16.png" rel="icon" sizes="16x16" type="image/png"/><link href="/manifest.json" rel="ma

In [15]:
manhwas = soup.find_all("div", class_="feature-cards")

In [16]:
print(len(manhwas))

0


In [14]:
manhwas

[]

In [19]:
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

In [26]:
driver = webdriver.Chrome()
driver.get("https://anilist.co/")

# Wait for page to load
wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "feature-cards")))

# Now you can scrape
soup = BeautifulSoup(driver.page_source, 'html.parser')
manhwas = soup.find_all("div", class_="feature-cards") 

print(f"Found {len(manhwas)} feature cards")

# Or use Selenium directly
elements = driver.find_elements(By.CLASS_NAME, "feature-cards")
for elem in elements:
    print(elem.text[:100]) 


Found 1 feature cards
Discover your obsessions
What are your highest rated genres or most watched voice actors? Follow you


In [1]:
!pip install webdriver-manager


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

def scrape_anilist_manhwa():
    # Setup driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    
    # Target the public landing page instead of /home
    driver.get("https://anilist.co/") 
    
    wait = WebDriverWait(driver, 15)
    try:
        # 1. Wait for the actual media card container elements to load
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cover")))
        
        # Scroll to ensure everything triggers/renders
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 4);")
        time.sleep(2)
        
        # 2. Update to the correct class structure on the landing page
        # AniList items typically sit inside cards with 'title' elements inside them
        cards = driver.find_elements(By.CLASS_NAME, "media-card")
        manhwa_data = []
        
        for card in cards[:20]:
            try:
                # Extract title text
                title_elem = card.find_element(By.CLASS_NAME, "title")
                title = title_elem.text.strip()
                
                # If titles are empty (sometimes happens before hover/interaction), 
                # grab the text directly or content attributes
                if not title:
                    title = card.text.split('\n')[0] 
                
                manhwa_data.append({
                    'Title': title,
                    'Type': 'Manga/Anime' # The landing page mixes both
                })
            except Exception as e:
                continue
                
        return pd.DataFrame(manhwa_data)
        
    finally:
        driver.quit()


In [4]:

# Run the scraper
df = scrape_anilist_manhwa()
print(df.head())

                                               Title         Type
0   Re:Zero kara Hajimeru Isekai Seikatsu 4th Season  Manga/Anime
1  Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu ...  Manga/Anime
2                          Tongari Boushi no Atelier  Manga/Anime
3                                          ONE PIECE  Manga/Anime
4  Class de 2-banme ni Kawaii Onnanoko to Tomodac...  Manga/Anime


In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument('--headless')  # Run without opening browser window
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
driver.get("https://anilist.co/")
print("Page loaded in headless mode")
driver.quit()

Page loaded in headless mode


In [7]:
print(df)
print(f"Total records: {len(df)}")

                                                Title         Type
0    Re:Zero kara Hajimeru Isekai Seikatsu 4th Season  Manga/Anime
1   Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu ...  Manga/Anime
2                           Tongari Boushi no Atelier  Manga/Anime
3                                           ONE PIECE  Manga/Anime
4   Class de 2-banme ni Kawaii Onnanoko to Tomodac...  Manga/Anime
5                                       MARRIAGETOXIN  Manga/Anime
6                           Tongari Boushi no Atelier  Manga/Anime
7    Re:Zero kara Hajimeru Isekai Seikatsu 4th Season  Manga/Anime
8                                      Yomi no Tsugai  Manga/Anime
9           Tensei Shitara Slime Datta Ken 4th Season  Manga/Anime
10  Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu ...  Manga/Anime
11               Tsue to Tsurugi no Wistoria Season 2  Manga/Anime
12       Mushoku Tensei III: Isekai Ittara Honki Dasu  Manga/Anime
13                                     Youjo Senki II  Manga/A

In [15]:
df.duplicated().sum()

np.int64(3)

In [16]:
df = df.drop_duplicates()

In [17]:
print(df)
print(f"Total unique records: {len(df)}")

                                                Title         Type
0    Re:Zero kara Hajimeru Isekai Seikatsu 4th Season  Manga/Anime
1   Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu ...  Manga/Anime
2                           Tongari Boushi no Atelier  Manga/Anime
3                                           ONE PIECE  Manga/Anime
4   Class de 2-banme ni Kawaii Onnanoko to Tomodac...  Manga/Anime
5                                       MARRIAGETOXIN  Manga/Anime
8                                      Yomi no Tsugai  Manga/Anime
9           Tensei Shitara Slime Datta Ken 4th Season  Manga/Anime
11               Tsue to Tsurugi no Wistoria Season 2  Manga/Anime
12       Mushoku Tensei III: Isekai Ittara Honki Dasu  Manga/Anime
13                                     Youjo Senki II  Manga/Anime
14                    Super no Ura de Yani Suu Futari  Manga/Anime
15             BLEACH: Sennen Kessen-hen - Kashin-tan  Manga/Anime
16    Mahou Shoujo Madoka☆Magica: Walpurgis no Kaiten  Manga/A

In [21]:
print(driver)

<selenium.webdriver.chrome.webdriver.WebDriver (session="69514293372e2b0f99153e4515ac91fa")>


In [23]:
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

driver.get("https://anilist.co/")

In [24]:
driver.title

'AniList: Track, Discover, Share Anime & Manga'

In [25]:
cards = driver.find_elements(By.CLASS_NAME, "media-card")

anime_links = []

for card in cards[:10]:
    try:
        link = card.find_element(By.TAG_NAME, "a").get_attribute("href")

        anime_links.append(link)

    except:
        continue

print(anime_links)

['https://anilist.co/anime/189046/ReZero-kara-Hajimeru-Isekai-Seikatsu-4th-Season/', 'https://anilist.co/anime/180745/Youkoso-Jitsuryoku-Shijou-Shugi-no-Kyoushitsu-e-4th-Season-2nenseihen-Ichi-Gakki/', 'https://anilist.co/anime/147105/Tongari-Boushi-no-Atelier/', 'https://anilist.co/anime/21/ONE-PIECE/', 'https://anilist.co/anime/169580/Class-de-2banme-ni-Kawaii-Onnanoko-to-Tomodachi-ni-Natta/', 'https://anilist.co/anime/199547/MARRIAGETOXIN/', 'https://anilist.co/anime/147105/Tongari-Boushi-no-Atelier/', 'https://anilist.co/anime/189046/ReZero-kara-Hajimeru-Isekai-Seikatsu-4th-Season/', 'https://anilist.co/anime/195600/Yomi-no-Tsugai/', 'https://anilist.co/anime/182205/Tensei-Shitara-Slime-Datta-Ken-4th-Season/']


In [26]:
for url in anime_links:
    driver.get(url)

    time.sleep(2)

    print(driver.title)

AniList
AniList
AniList
AniList
AniList
AniList
AniList
AniList
AniList
AniList


In [27]:
title = driver.find_element(
    By.CLASS_NAME,
    "title"
).text

print(title)

Tensei Shitara Slime Datta Ken


In [29]:
description = driver.find_element(
    By.CLASS_NAME,
    "description"
).text

print(description[:200])

The fourth season of Tensei Shitara Slime Datta Ken.

Demon Lord Rimuru's dream of creating an alliance between humans and monsters takes a step closer to being realized. As Tempest continues to prosp


In [34]:
rows = driver.find_elements(By.CLASS_NAME, "data-set")

print(f"Found {len(rows)} rows")

for row in rows:
    try:
        label = row.find_element(By.CLASS_NAME, "type").text
        value = row.find_element(By.CLASS_NAME, "value").text

        print(f"{label}: {value}")
    except:
        pass

Found 19 rows
Airing: Ep 11: 1d 21h 13m
Format: TV
Episode Duration: 24 mins
Status: Releasing
Start Date: Apr 3, 2026
Season: Spring 2026
Average Score: 81%
Mean Score: 82%
Popularity: 101863
Favorites: 2946
Studios: 8-bit
Producers: Bandai Namco Filmworks
Kodansha
Micro Magazine Sha
Bandai Namco Music Live
BANDAI SPIRITS
BS11
ADK Marketing Solutions
Sony Music Solutions
8-bit
Nippon Television Network
Source: Light Novel
Hashtag: #転スラ #tensura
Genres: Action
Adventure
Comedy
Fantasy
Romaji: Tensei Shitara Slime Datta Ken 4th Season
English: That Time I Got Reincarnated as a Slime Season 4
Native: 転生したらスライムだった件 第4期
Synonyms: Tensura 4
転スラ 4
เกิดใหม่ทั้งทีก็เป็นสไลม์ไปซะแล้ว ซีซั่น 4
О моём перерождении в слизь 4
Aquella vez que me convertí en slime - Temporada 4


In [35]:
rows = driver.find_elements(By.CLASS_NAME, "data-set")

for row in rows:
    try:
        label = row.find_element(By.CLASS_NAME, "type").text

        if label == "Genres":
            genres = row.find_elements(By.TAG_NAME, "span")

            genre_list = [g.text for g in genres]

            print(genre_list)

    except:
        pass

['Action', 'Adventure', 'Comedy', 'Fantasy']


In [39]:
rows = driver.find_elements(By.CLASS_NAME, "data-set")

anime_info = {}

for row in rows:
    try:
        label = row.find_element(By.CLASS_NAME, "type").text
        value = row.find_element(By.CLASS_NAME, "value").text

        anime_info[label] = value

    except:
        pass

print(anime_info)

{'Airing': 'Ep 12: 55d 20h 10m', 'Format': 'TV', 'Episodes': '19', 'Episode Duration': '24 mins', 'Status': 'Releasing', 'Start Date': 'Apr 8, 2026', 'Season': 'Spring 2026', 'Average Score': '88%', 'Mean Score': '88%', 'Popularity': '108677', 'Favorites': '4158', 'Studios': 'WHITE FOX', 'Producers': 'KADOKAWA\nHakuhodo DY Music & Pictures\nAT-X\nDAXEL\nMemory-Tech', 'Source': 'Light Novel', 'Hashtag': '#rezero #リゼロ', 'Genres': 'Action\nAdventure\nDrama\nFantasy\nPsychological\nRomance\nThriller', 'Romaji': 'Re:Zero kara Hajimeru Isekai Seikatsu 4th Season', 'English': 'Re:ZERO -Starting Life in Another World- Season 4', 'Native': 'Re:ゼロから始める異世界生活 4th season', 'Synonyms': 'Re:ZERO รีเซทชีวิต ฝ่าวิกฤตต่างโลก ซีซั่น 4\nRe:ZERO – Жизнь с нуля в альтернативном мире 4'}


In [45]:
anime_data = []

for url in anime_links[:3]:
    print(url)
    driver.get(url)

    title = driver.find_element(By.CLASS_NAME, "romaji").text

    print("TITLE FOUND:", title)

    anime_data.append({
        "Title": title,
        "Genres": ", ".join(genre_list),
        "Average Score": anime_info.get("Average Score"),
        "Mean Score": anime_info.get("Mean Score"),
        "Popularity": anime_info.get("Popularity"),
        "Favorites": anime_info.get("Favorites")
    })

https://anilist.co/anime/189046/ReZero-kara-Hajimeru-Isekai-Seikatsu-4th-Season/


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":".romaji"}
  (Session info: chrome=149.0.7827.114); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff750253fa5+14925]
	chromedriver!GetHandleVerifier [0x7ff750254000+14980]
	chromedriver!(No symbol) [0x7ff74fd9793d]
	chromedriver!(No symbol) [0x7ff74fdf1aad]
	chromedriver!(No symbol) [0x7ff74fdf1dac]
	chromedriver!(No symbol) [0x7ff74fe427d7]
	chromedriver!(No symbol) [0x7ff74fe3f39b]
	chromedriver!(No symbol) [0x7ff74fde401c]
	chromedriver!(No symbol) [0x7ff74fde4f43]
	chromedriver!GetHandleVerifier [0x7ff750837591+5f7f11]
	chromedriver!GetHandleVerifier [0x7ff750831902+5f2282]
	chromedriver!GetHandleVerifier [0x7ff750857115+617a95]
	chromedriver!GetHandleVerifier [0x7ff750271dce+3274e]
	chromedriver!GetHandleVerifier [0x7ff75027a82c+

In [42]:
print(anime_links[:3])

['https://anilist.co/anime/189046/ReZero-kara-Hajimeru-Isekai-Seikatsu-4th-Season/', 'https://anilist.co/anime/180745/Youkoso-Jitsuryoku-Shijou-Shugi-no-Kyoushitsu-e-4th-Season-2nenseihen-Ichi-Gakki/', 'https://anilist.co/anime/147105/Tongari-Boushi-no-Atelier/']


In [44]:
anime_data

[{'Title': 'Tensei Shitara Slime Datta Ken',
  'Genres': 'Action, Adventure, Comedy, Fantasy',
  'Average Score': '88%',
  'Mean Score': '88%',
  'Popularity': '108677',
  'Favorites': '4158'},
 {'Title': 'Tensei Shitara Slime Datta Ken',
  'Genres': 'Action, Adventure, Comedy, Fantasy',
  'Average Score': '88%',
  'Mean Score': '88%',
  'Popularity': '108677',
  'Favorites': '4158'},
 {'Title': 'Tensei Shitara Slime Datta Ken',
  'Genres': 'Action, Adventure, Comedy, Fantasy',
  'Average Score': '88%',
  'Mean Score': '88%',
  'Popularity': '108677',
  'Favorites': '4158'}]

In [46]:
print(driver.title)

AniList


In [47]:
driver.get(url)

In [48]:
for i, link in enumerate(anime_links[:5]):
    print(i, repr(link))

0 'https://anilist.co/anime/189046/ReZero-kara-Hajimeru-Isekai-Seikatsu-4th-Season/'
1 'https://anilist.co/anime/180745/Youkoso-Jitsuryoku-Shijou-Shugi-no-Kyoushitsu-e-4th-Season-2nenseihen-Ichi-Gakki/'
2 'https://anilist.co/anime/147105/Tongari-Boushi-no-Atelier/'
3 'https://anilist.co/anime/21/ONE-PIECE/'
4 'https://anilist.co/anime/169580/Class-de-2banme-ni-Kawaii-Onnanoko-to-Tomodachi-ni-Natta/'


In [49]:
driver.get(url)

time.sleep(3)

print("Current URL:", driver.current_url)
print("Title:", driver.title)

Current URL: https://anilist.co/anime/189046/ReZero-kara-Hajimeru-Isekai-Seikatsu-4th-Season/
Title: AniList
